# M4 · Sistemes de Big Data — RA3
## Notebook guiado: Apache Spark con PySpark en Google Colab

**Caso conductor:** StreamNow  
**Tipo:** laboratorio guiado, no evaluable

### Objetivos
Al terminar este notebook deberías poder:

- crear una sesión de Spark en Colab;
- trabajar con **DataFrames**;
- distinguir **transformations** y **actions**;
- comprobar la **lazy evaluation**;
- entender el papel de las **particiones**;
- identificar operaciones que pueden provocar **shuffle**;
- usar **Spark SQL**;
- comparar **CSV y Parquet**;
- interpretar un plan de ejecución con `explain()`;
- usar `cache()` con criterio.

> **Importante:** Google Colab ejecuta Spark en una sola máquina. Aquí observaremos el modelo de ejecución de Spark, pero **no es un benchmark de un clúster real**.

## 0. Preparación del entorno

Colab no incluye PySpark por defecto. Lo instalamos y creamos una sesión local de Spark.

`local[*]` indica que Spark podrá utilizar los núcleos disponibles en la máquina de Colab.

In [ ]:
!pip -q install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("M4_RA3_StreamNow")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

## 1. Creamos el dataset StreamNow

Cada fila representa un evento de reproducción:

- `event_id`
- `user_id`
- `content_id`
- `timestamp`
- `country`
- `device`
- `duration`
- `quality`
- `buffering_ms`

Generaremos los datos directamente con Spark.

In [ ]:
N = 200_000

events = (
    spark.range(N)
    .withColumnRenamed("id", "event_id")
    .withColumn("user_id", F.concat(F.lit("U"), F.lpad((F.col("event_id") % 25_000).cast("string"), 5, "0")))
    .withColumn("content_id", F.concat(F.lit("C"), F.lpad((F.col("event_id") % 800).cast("string"), 4, "0")))
    .withColumn(
        "timestamp",
        F.to_timestamp(
            F.from_unixtime(
                F.unix_timestamp(F.lit("2026-01-01 00:00:00")) + F.col("event_id") * 20
            )
        )
    )
    .withColumn(
        "country",
        F.when(F.rand(seed=1) < 0.55, "ES")
         .when(F.rand(seed=2) < 0.45, "FR")
         .when(F.rand(seed=3) < 0.50, "DE")
         .when(F.rand(seed=4) < 0.50, "IT")
         .otherwise("PT")
    )
    .withColumn(
        "device",
        F.when(F.rand(seed=5) < 0.40, "mobile")
         .when(F.rand(seed=6) < 0.50, "tv")
         .otherwise("desktop")
    )
    .withColumn("duration", (F.rand(seed=7) * 7200 + 60).cast("int"))
    .withColumn(
        "quality",
        F.when(F.rand(seed=8) < 0.60, "1080p")
         .when(F.rand(seed=9) < 0.75, "720p")
         .otherwise("4K")
    )
    .withColumn("buffering_ms", (F.rand(seed=10) * 4000).cast("int"))
)

events.printSchema()

In [ ]:
events.show(5, truncate=False)

### Primera observación

Aunque vemos una tabla parecida a Pandas, este objeto es un **Spark DataFrame**.

Spark representa el dataset como un conjunto de particiones que pueden ser procesadas en paralelo.

In [ ]:
print("Número de particiones actuales:", events.rdd.getNumPartitions())

## 2. Transformations vs Actions

Una **transformation** describe qué queremos hacer.

Ejemplos:
- `select()`
- `filter()`
- `withColumn()`
- `groupBy()`
- `join()`

Una **action** pide un resultado o fuerza una escritura.

Ejemplos:
- `count()`
- `show()`
- `collect()`
- `write...`

Spark usa **lazy evaluation**: normalmente no ejecuta las transformations hasta que aparece una action.

In [ ]:
events_es = events.filter(F.col("country") == "ES")
events_es_mobile = events_es.filter(F.col("device") == "mobile")
selected = events_es_mobile.select("content_id", "duration", "quality")

print(type(selected))
print("Todavía solo hemos definido transformaciones.")

Ahora lanzamos una **action**.

In [ ]:
selected.show(10)

### Mini-reto 1

Crea un DataFrame llamado `long_sessions` que contenga únicamente:

- `user_id`
- `content_id`
- `country`
- `duration`

para eventos con duración superior a **3600 segundos**.

Después muestra 5 filas.

In [ ]:
long_sessions = (
    events
    .filter(F.col("duration") > 3600)
    .select("user_id", "content_id", "country", "duration")
)

long_sessions.show(5)

## 3. Lazy evaluation

Construimos una cadena de operaciones sin pedir todavía ningún resultado.

In [ ]:
plan_demo = (
    events
    .filter(F.col("country") == "ES")
    .select("content_id", "duration")
    .groupBy("content_id")
    .agg(F.sum("duration").alias("total_duration"))
)

print("Plan definido. Aún no hemos pedido ningún resultado.")

Ahora usamos `explain()`.

Esto no ejecuta el análisis completo, pero permite ver cómo Spark ha organizado el plan.

In [ ]:
plan_demo.explain(mode="formatted")

Y ahora sí lanzamos una action:

In [ ]:
plan_demo.orderBy(F.desc("total_duration")).show(10)

## 4. Particiones y paralelismo

Una **partición Spark** es una unidad lógica de datos que normalmente será procesada por una **Task**.

En un clúster real:

- muchas particiones permiten paralelismo;
- demasiadas particiones generan overhead;
- muy pocas pueden desaprovechar recursos.

En Colab seguimos viendo las particiones, aunque todo ocurra en una sola máquina.

In [ ]:
print("Particiones iniciales:", events.rdd.getNumPartitions())

events_8 = events.repartition(8)
print("Después de repartition(8):", events_8.rdd.getNumPartitions())

events_4 = events_8.coalesce(4)
print("Después de coalesce(4):", events_4.rdd.getNumPartitions())

### `repartition()` vs `coalesce()`

- `repartition(n)` redistribuye datos y normalmente implica shuffle.
- `coalesce(n)` se usa sobre todo para **reducir** particiones intentando evitar una redistribución completa.

No existe un número mágico de particiones válido para todos los casos.

## 5. Narrow vs Wide Transformations

### Narrow
Cada partición puede trabajar de forma relativamente independiente.

Ejemplos:
- `filter`
- `select`
- `withColumn`

### Wide
Los datos deben reorganizarse entre particiones.

Ejemplos típicos:
- `groupBy`
- `orderBy`
- `distinct`
- muchos `join`

Ese movimiento de datos se denomina **shuffle**.

In [ ]:
narrow_demo = (
    events
    .filter(F.col("duration") > 1000)
    .select("country", "device", "duration")
)

narrow_demo.explain(mode="formatted")

In [ ]:
wide_demo = (
    events
    .groupBy("country")
    .agg(F.sum("duration").alias("total_duration"))
)

wide_demo.explain(mode="formatted")

### ¿Dónde vemos el shuffle?

En el plan de una agregación como `groupBy`, busca un nodo parecido a:

`Exchange`

Esa palabra suele indicar una redistribución de datos entre particiones.

## 6. Shuffle en acción

Pregunta de negocio:

> ¿Cuánto tiempo total de reproducción acumula cada país?

In [ ]:
country_watchtime = (
    events
    .groupBy("country")
    .agg(
        F.sum("duration").alias("total_duration"),
        F.count("*").alias("events")
    )
    .orderBy(F.desc("total_duration"))
)

country_watchtime.show()

### Data Skew

Nuestro dataset no está perfectamente equilibrado: España aparece con mucha más frecuencia.

Eso permite recordar:

> Si una clave concentra una gran parte de los datos, una partición puede convertirse en cuello de botella después de un shuffle.

In [ ]:
events.groupBy("country").count().orderBy(F.desc("count")).show()

## 7. Spark SQL

Los mismos datos se pueden consultar mediante la API de DataFrames o mediante SQL.

Primero registramos una vista temporal.

In [ ]:
events.createOrReplaceTempView("events")

In [ ]:
spark.sql("""
SELECT
    country,
    device,
    COUNT(*) AS events,
    ROUND(SUM(duration) / 3600, 1) AS hours_watched
FROM events
GROUP BY country, device
ORDER BY hours_watched DESC
""").show(20)

### Mini-reto 2

Con **Spark SQL**, calcula por calidad (`quality`):

- número de eventos;
- duración media;
- buffering medio.

Ordena por número de eventos descendente.

In [ ]:
spark.sql("""
SELECT
    quality,
    COUNT(*) AS events,
    ROUND(AVG(duration), 1) AS avg_duration,
    ROUND(AVG(buffering_ms), 1) AS avg_buffering_ms
FROM events
GROUP BY quality
ORDER BY events DESC
""").show()

## 8. Guardamos el mismo dataset como CSV y Parquet

Vamos a escribir el mismo dataset en:

- CSV
- Parquet

> No buscamos hacer un benchmark científico. El objetivo es observar diferencias de estructura, esquema y plan de lectura.

In [ ]:
import shutil

base = "/content/streamnow_lab"
csv_path = f"{base}/events_csv"
parquet_path = f"{base}/events_parquet"

shutil.rmtree(base, ignore_errors=True)

events.write.mode("overwrite").option("header", True).csv(csv_path)
events.write.mode("overwrite").parquet(parquet_path)

print("Datos guardados en:")
print(csv_path)
print(parquet_path)

## 9. CSV: esquema y lectura

CSV es texto plano. Si no proporcionamos un esquema, Spark puede necesitar inferir tipos o leerlos como texto.

In [ ]:
csv_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)

csv_df.printSchema()

## 10. Parquet: esquema embebido

Parquet almacena información de esquema junto con los datos.

In [ ]:
parquet_df = spark.read.parquet(parquet_path)
parquet_df.printSchema()

## 11. Projection Pushdown

Supongamos que solo necesitamos:

- `country`
- `duration`

Con Parquet, Spark puede solicitar únicamente esas columnas.

In [ ]:
projection = (
    spark.read.parquet(parquet_path)
    .select("country", "duration")
)

projection.explain(mode="formatted")

Busca en el plan la sección relacionada con el escaneo Parquet.

Deberías observar que el esquema leído contiene únicamente las columnas necesarias.

Eso es **Projection Pushdown**:

> evitar leer columnas que la consulta no necesita.

## 12. Predicate Pushdown

Ahora añadimos un filtro.

In [ ]:
predicate = (
    spark.read.parquet(parquet_path)
    .filter(F.col("country") == "ES")
    .select("country", "duration")
)

predicate.explain(mode="formatted")

Busca elementos como:

- `DataFilters`
- `PushedFilters`

Spark puede trasladar parte del filtro hacia la lectura del archivo.

Con formatos columnares como Parquet y sus metadatos, esto puede reducir los datos que deben procesarse.

## 13. CSV vs Parquet: inspección del plan

Hacemos la misma consulta sobre ambos formatos.

In [ ]:
csv_query = (
    csv_df
    .filter(F.col("country") == "ES")
    .select("country", "duration")
)

parquet_query = (
    parquet_df
    .filter(F.col("country") == "ES")
    .select("country", "duration")
)

print("=== CSV ===")
csv_query.explain(mode="formatted")

print("\n=== PARQUET ===")
parquet_query.explain(mode="formatted")

### Qué deberías llevarte

**CSV**
- texto;
- orientado a filas;
- esquema no embebido;
- útil para intercambio y datasets sencillos.

**Parquet**
- binario;
- columnar;
- esquema embebido;
- especialmente útil para analítica a escala.

## 14. Cache: evitar recalcular lo mismo

`cache()` no ejecuta inmediatamente el DataFrame.

Primero **marca** el resultado para ser cacheado.

La primera action lo calcula y permite reutilizarlo posteriormente.

In [ ]:
es_events = events.filter(F.col("country") == "ES")

es_events.cache()
print("Marcado para cache.")

# Primera action: calcula el DataFrame y materializa la caché
print("Filas ES:", es_events.count())

In [ ]:
es_events.groupBy("device").count().show()

es_events.groupBy("quality").agg(
    F.avg("buffering_ms").alias("avg_buffering_ms")
).show()

In [ ]:
es_events.unpersist()

## 15. Ejercicio guiado final — StreamNow

La dirección técnica quiere responder:

> **¿Qué combinación de país y dispositivo concentra más horas de reproducción y qué nivel medio de buffering presenta?**

Debes:

1. agrupar por `country` y `device`;
2. calcular:
   - número de eventos;
   - horas totales;
   - buffering medio;
3. ordenar por horas totales de mayor a menor;
4. mostrar los 10 primeros resultados;
5. observar el plan con `explain()`;
6. identificar si existe shuffle.

In [ ]:
final_analysis = (
    events
    .groupBy("country", "device")
    .agg(
        F.count("*").alias("events"),
        (F.sum("duration") / 3600).alias("hours_watched"),
        F.avg("buffering_ms").alias("avg_buffering_ms")
    )
    .orderBy(F.desc("hours_watched"))
)

final_analysis.show(10, truncate=False)

In [ ]:
final_analysis.explain(mode="formatted")

### Preguntas de reflexión

1. ¿Qué parte del análisis provoca el shuffle?
2. ¿Por qué `filter()` normalmente no necesita redistribuir todos los datos?
3. ¿Qué diferencia hay entre una partición de almacenamiento y una partición interna de Spark?
4. ¿Por qué Parquet puede ser más eficiente que CSV si solo necesitamos 2 de 9 columnas?
5. ¿Por qué no tendría sentido hacer `collect()` sobre millones de filas?
6. ¿Cuándo tendría sentido utilizar `cache()`?

## 16. Resumen final

Hemos recorrido:

**DataFrame → Transformations → Lazy Evaluation → Action → Job / Stage / Task → Partitions → Shuffle → Spark SQL → Parquet → Pushdown → Cache**

### Ideas clave

- Spark **procesa** datos; no es necesariamente donde se almacenan.
- Un DataFrame puede estar dividido en múltiples particiones.
- Las transformations construyen un plan; las actions provocan ejecución.
- `groupBy`, `orderBy` y muchos `join` pueden provocar shuffle.
- El shuffle es costoso porque redistribuye datos.
- Parquet es especialmente adecuado para analítica.
- `explain()` permite observar cómo Spark piensa ejecutar una consulta.
- Cachear puede ayudar cuando reutilizamos un resultado costoso, pero no debe hacerse por defecto.

### Siguiente paso

En el laboratorio de **Databricks** veremos estos mismos conceptos dentro de una plataforma profesional de datos y trabajaremos también con tablas y Delta Lake.